# Attention Mechanisms and Transformers

<a target="_blank" href="https://colab.research.google.com/github/imamitjain/notebooks/blob/main/04-llm-and-transformers/01_attention_and_transformers.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objective:** Understand the transformer architecture from the ground up — self-attention, multi-head attention, positional encoding, and why transformers dominate modern AI.

**Prerequisites:** RNNs and sequence models (Section 03)

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q numpy matplotlib torch


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

## 1. The Intuition Behind Attention

In a sequence, not all tokens are equally important to each other. Attention lets the model learn *which* parts of the input to focus on for each output position.

**Key insight:** Instead of compressing the entire input into a fixed-size hidden state (like RNNs), attention creates a weighted combination of all input positions.

## 2. Scaled Dot-Product Attention

The core computation: $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    weights = F.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    return output, weights

In [ ]:
torch.manual_seed(42)
seq_len, d_model = 5, 8

Q = torch.randn(1, seq_len, d_model)
K = torch.randn(1, seq_len, d_model)
V = torch.randn(1, seq_len, d_model)

output, weights = scaled_dot_product_attention(Q, K, V)

print(f"Input shape:   {Q.shape}")
print(f"Output shape:  {output.shape}")
print(f"Weights shape: {weights.shape}")

plt.figure(figsize=(6, 5))
plt.imshow(weights[0].detach().numpy(), cmap='Blues')
plt.colorbar()
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.title("Attention Weights")
plt.show()

## 3. Multi-Head Attention

Multiple attention heads let the model attend to different types of relationships simultaneously.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)

        Q = self.W_q(Q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(K).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(V).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

        output, weights = scaled_dot_product_attention(Q, K, V, mask)

        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.W_o(output), weights

In [ ]:
mha = MultiHeadAttention(d_model=64, n_heads=8)
x = torch.randn(2, 10, 64)
output, weights = mha(x, x, x)

print(f"Input:   {x.shape}")
print(f"Output:  {output.shape}")
print(f"Weights: {weights.shape}  (batch, heads, seq, seq)")

## 4. Positional Encoding

Transformers have no built-in notion of order. Positional encodings inject sequence position information.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [ ]:
pe = PositionalEncoding(d_model=64)
encodings = pe.pe[0, :100, :].numpy()

plt.figure(figsize=(12, 4))
plt.imshow(encodings.T, aspect='auto', cmap='RdBu')
plt.colorbar()
plt.xlabel("Position")
plt.ylabel("Dimension")
plt.title("Positional Encodings (first 100 positions)")
plt.show()

## 5. Transformer Block

Putting it all together: multi-head attention + feed-forward network + residual connections + layer norm.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, _ = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))
        return x

block = TransformerBlock(d_model=64, n_heads=8, d_ff=256)
x = torch.randn(2, 10, 64)
out = block(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")
print(f"Parameters: {sum(p.numel() for p in block.parameters()):,}")

## Try It Yourself

1. Add a **causal mask** to the attention to prevent tokens from attending to future positions (used in GPT-style models). Verify that the attention weights are lower-triangular.
2. Stack 4 transformer blocks and pass the same input through. Track how the output changes at each layer (compute the L2 norm of the difference between consecutive layers).
3. Experiment with the number of heads: try 1, 2, 4, 8 heads with `d_model=64`. Do more heads always help? Train on a simple sequence task to compare.

In [ ]:
# Your code here